# Session 5: Classify images with AI

This notebook demonstrates how to find, access, and deploy an AI model to process underwater imagery.  

## Step 1: Set up tools

This notebook depends on the ultralytics and torch packages.

In [ ]:
# If the ultralytics package has not yet been installed, we'll install it.
try:
    import ultralytics
except ImportError:
    !pip install -q ultralytics


# The torch package is a dependency of the ultralytics package, and thus
# should already be installed. However it is good practice to confirm with a
# try/except.
try:
    import torch
except ImportError:
    !pip install -q torch

# Additional libraries are imported in later cells

First, we'll inspect our GPU environment. The GPU you get on the OOI JupyterHub's PyTorch notebook can vary from session to session. For example, you may see 1 NVIDIA L40S GPU with ~44GiB of VRAM, or 1 NVIDIA H100 GPU with ~93GiB of VRAM.

If you're using Google Colab, you'll likely see 1 Tesla T4 GPU with ~14.5GiB of VRAM.

In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("PyTorch version:", torch.__version__)
print("PyTorch CUDA version:", torch.version.cuda)

def bytes_to_gib(num_bytes: int) -> float:
    return num_bytes / 1024**3

if torch.cuda.is_available():
    num_devs = torch.cuda.device_count()
    print(f"Number of GPUs: {num_devs}")
    dev_num = torch.cuda.current_device()
    gpu_type = torch.cuda.get_device_name(dev_num)
    print(f"Current GPU: {gpu_type}")
    print("\nGPU details:")
    for dev in range(num_devs):
        properties = torch.cuda.get_device_properties(dev)
        free_mem, total_mem = torch.cuda.mem_get_info(dev)
        print(f"\nGPU {dev}: {properties.name}")
        print(f"  Total VRAM:          {bytes_to_gib(properties.total_memory):.2f} GiB")
else:
    print("No GPU found, using CPU")

# Part A
## 1 - Discover - Find data and pre-trained model

We'll be using the SHR (Southern Hydrate Ridge) DSCAM (Digital Still Camera) data from OOI. 

We'll be running the pretrained YOLOv11 model developed for these images as described in this paper: https://aslopubs.onlinelibrary.wiley.com/doi/10.1002/lom3.70057. 

Our next step is to find this pre-trained model to run inference on our imagery. The model card is located here: https://huggingface.co/atticus-carter/SHR_DSCAM.

## 2 - Access

Using the ultralytics package, we can download the model directly from the Hugging Face URL to use it in Python.

The model is about 130MB and will be downloaded into your notebook space; the message printed below the cell shows where the weights file (`best.pt`) was saved.

We also use `settings.update` to keep the folders ultralytics creates for this session under `./session_5_outputs/`. Note that these settings are saved to your ultralytics configuration, so they also apply to other notebooks you run in this environment.

In [ ]:
from ultralytics import YOLO
from ultralytics import settings
# Update default directories
settings.update({
    'datasets_dir': './session_5_outputs/datasets',
    'weights_dir': './session_5_outputs/weights',
    'runs_dir': './session_5_outputs/runs',
})

yolo_model = YOLO("https://huggingface.co/atticus-carter/SHR_DSCAM/resolve/main/best.pt")

## 3 - Wrangle

In [ ]:
# On the OOI JupyterHub, read the image directly from the data-proximate file system
data_path = "/home/jovyan/ooi/rsn_cabled/rsn_data/DVT_Data/mj01b/CAMDSB103_10.33.7.5/2022/11/04/CAMDSB103_10.33.7.5_20221104T004544_UTC.jpg"

# On Google Colab, uncomment the line below to use the image URL from the OOI raw data server instead
#data_path = "https://rawdata.oceanobservatories.org/files/RS01SUM2/MJ01B/CAMDSB103/2022/11/04/CAMDSB103_10.33.7.5_20221104T004544_UTC.jpg"

Before we run the model, let's look at the raw image ourselves. Note the image's size and aspect ratio: the model will resize it so its longest side is `imgsz=1024` pixels, so small or distant animals end up covering only a few pixels.

In [ ]:
from io import BytesIO
import requests
from PIL import Image

scale = 0.25  # display images at 25% size

def show_raw_image(path):
    """Open an image from a file path or URL, print its size, and display it."""
    if str(path).startswith("http"):
        response = requests.get(path)
        response.raise_for_status()
        raw_img = Image.open(BytesIO(response.content))
    else:
        raw_img = Image.open(path)

    print(f"Image size:   {raw_img.width} x {raw_img.height} pixels")
    print(f"Aspect ratio: {raw_img.width / raw_img.height:.2f}")

    display(
        raw_img.resize(
            (int(raw_img.width * scale),
             int(raw_img.height * scale))
        )
    )

show_raw_image(data_path)

**Before running the model:** What animals do you see? How many? Which do you expect the model to detect, and which might it miss?

## 4 - Reuse
To test that the model has been properly initialized, we'll first run it on an image from within the time period the model was trained on. Per the paper, the model was trained on imagery from the 2022-2023 DSCAM deployment, and the image we just looked at is from November 2022.

Each of the arguments for the `predict` command is:
- source: The input data to run inference on.
- save: Whether to save our output inference to disk.
- save_txt: Whether to save the detection outputs to txt files.
- save_conf: Whether to save the confidence scores of each detection in the detection txt files.
- imgsz: The image size to resize each incoming image to. Taken from the model arguments for this model here: https://huggingface.co/atticus-carter/SHR_DSCAM/blob/main/args.yaml.
- conf: The confidence score threshold -- all detections below this confidence value will be ignored. There is no "correct" value here -- 0.6 is chosen as a medium value.

In [ ]:
results = yolo_model.predict(
    source=data_path,
    save=True,
    save_txt=True,
    save_conf=True,
    imgsz=1024,
    conf=0.6,
)
img = results[0].plot(pil=True)

display(
    img.resize(
        (int(img.width * scale),
         int(img.height * scale))
    )
)

Let's look at the label format that YOLO creates.

In [ ]:
from pathlib import Path

# Each predict() run saves to a new folder (session_5_outputs/runs/detect/predict, predict2, ...),
# so build the label path from this run's save_dir and the image file name.
save_dir = Path(results[0].save_dir)
image_name = Path(results[0].path).stem
example_label_file_path = save_dir / "labels" / f"{image_name}.txt"

with open(example_label_file_path, 'r') as f:
    content = f.read()
    print(f"Opened: {example_label_file_path}")
    print(content)

In the YOLO data format, each image that has had inference run on it has a text file containing its annotations.

Each line in the text file represents a detection. Each line has the format:

`class_id center_x center_y width height confidence`

- `center_x`, `center_y`, `width`, and `height` describe the bounding box. They are normalized to the range 0–1 as fractions of the image's width and height, so they don't depend on the image's pixel size.
- `confidence` is only included because we set `save_conf=True` when running `predict`.

As you can see in the previous cell output, the model has detected three instances of class `10`. To understand which class `10` is referring to, we'll need to check the yolo model's class names.

In [ ]:
yolo_model.names

In the cell above, we see that `10` is the class index for 'Sebastes'. Note that the class indexes are zero-indexed.

# Part B
In Part A we ran the model on an image from the period it was trained on. Now we'll run it on imagery from outside its training set, and see how its confidence and accuracy change as the imagery gets less like the training data.

## Recent Southern Hydrate Ridge imagery
### 1 - Discover

The following image is from the same DSCAM at the SHR, but is from December 2025. You can see that the camera is now pointed at a different feature.

### 2 - Access

This image is on the OOI JupyterHub, just like the image in Part A.

In [ ]:
# On the OOI JupyterHub, read the image directly from the data-proximate file system
shr_2025_path = "/home/jovyan/ooi/rsn_cabled/rsn_data/DVT_Data/mj01b/CAMDSB103_10.33.7.5/2025/12/01/CAMDSB103_10.33.7.5_20251201T021648_UTC.jpg"

# On Google Colab, uncomment the line below to use the image URL from the OOI raw data server instead
#shr_2025_path = "https://rawdata.oceanobservatories.org/files/RS01SUM2/MJ01B/CAMDSB103/2025/12/01/CAMDSB103_10.33.7.5_20251201T021648_UTC.jpg"

### 3 - Wrangle

As in Part A, let's look at the raw image before running the model. How does it compare with the Part A image?

In [ ]:
show_raw_image(shr_2025_path)

### 4 - Reuse

Next, we'll run the model on this image, which is from outside the model's training set, using the same `predict` arguments as in Part A.

In [ ]:
results = yolo_model.predict(
    source=shr_2025_path,
    save=True,
    save_txt=True,
    save_conf=True,
    imgsz=1024,
    conf=0.6,
)
img = results[0].plot(pil=True)
display(
    img.resize(
        (int(img.width * scale),
         int(img.height * scale))
    )
)

Notably, the confidence scores are slightly lower than when we ran the model on imagery from its training set. As the input data differs from a model's training data, the model may have lower confidence and lower accuracy.

## ROV Jason imagery
### 1 - Discover

We'll next run the model on some data from the ROV Jason: an image from the SulisCam on dive J2-1715, from the WHOI Sealog server. This is a different camera entirely, and imagery that the original model was not trained on.

### 2 - Access

First, we'll download the image from the WHOI Sealog server and save it as a file in our notebook environment.

As shown in the previous cells, the `predict` function in the ultralytics package can run directly on image URLs. When running an image URL, the function will download the image to the notebook's directory. When running directly on files in the JupyterHub, no download is required for the data-proximate computing.

However, if we wanted to explicitly fetch the image URL and save it under a different name, we could use the `requests` package to fetch the image, and then give the local path to the `predict` function.

In [ ]:
from pathlib import Path
import requests

# Image URL
url = "https://sealog.whoi.edu/sealog-jason/server/files/images/J2-1715/SulisCam/SulisCam.20250813_030949000.jpg"

jason_path = Path("rov_image.jpg") # for demonstration purposes save the image under a different name

response = requests.get(url)
response.raise_for_status()

with open(jason_path, "wb") as f:
    f.write(response.content)

### 3 - Wrangle

Let's look at the raw image. Compare its size and aspect ratio with the SHR images. How different does it look from what the model was trained on?

In [ ]:
show_raw_image(jason_path)

### 4 - Reuse

Now that the image is downloaded, we can run inference on it.

In [ ]:
results = yolo_model.predict(
    source=jason_path,
    save=True,
    save_txt=True,
    save_conf=True,
    imgsz=1024,
    conf=0.6,
)
img = results[0].plot(pil=True)
display(
    img.resize(
        (int(img.width * scale),
         int(img.height * scale))
    )
)

**Compare the results:** How do the detections and confidence scores on the Jason image compare with the two SHR images? Are the labels the model assigned plausible for this scene? What does this suggest about using a model on imagery from a different camera or location than it was trained on?

# Wrapping up Session

Let's get some information we can use in our documentation.

## When did you access your data?

In a notebook cell it's a good idea to print out the date (or datetime) when you ran your code. That way you have it for when you ever need that information in your documentation.

In [ ]:
from datetime import datetime

print("Run datetime: " + datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

## What library versions did you use?

Below we are printing the package versions ran with this notebook in case we ever need to look back at this run and know when and what we were working with.

In [ ]:
from importlib.metadata import version

print("\nThis notebook was run with:")

print("ultralytics version: " + version("ultralytics"))
print("torch version: " + version("torch"))
print("pillow version: " + version("Pillow"))
print("requests version: " + version("requests"))

# Above we just asked for main package versions, but even better is saving a
#     requirements.txt file which you can write with:
# !pip freeze > requirements.txt
#    will write you a file with ALL versions of everything in your environment

## Record your data citations

*To be completed.*

## Check data licenses for your purpose

*To be completed.*